# Rice Leaf Disease Detection
Ready-to-run template. Update dataset path if needed.

In [ ]:
import os, zipfile
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras import layers, models

zip_path='PRCP-1001-RiceLeaf.zip'  # change if needed
extract_dir='riceleaf_data'
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path) as z: z.extractall(extract_dir)

# find dataset root containing class folders
root=extract_dir
for r,ds,fs in os.walk(extract_dir):
    if len(ds)>=3:
        root=r; break
print(root)

train_ds=image_dataset_from_directory(root,validation_split=0.2,subset='training',seed=42,image_size=(224,224),batch_size=32)
val_ds=image_dataset_from_directory(root,validation_split=0.2,subset='validation',seed=42,image_size=(224,224),batch_size=32)
class_names=train_ds.class_names

AUTOTUNE=tf.data.AUTOTUNE
train_ds=train_ds.prefetch(AUTOTUNE)
val_ds=val_ds.prefetch(AUTOTUNE)

aug=tf.keras.Sequential([
layers.RandomFlip('horizontal'),
layers.RandomRotation(0.1),
layers.RandomZoom(0.1)
])

base=tf.keras.applications.MobileNetV2(input_shape=(224,224,3),include_top=False,weights='imagenet')
base.trainable=False
inp=layers.Input((224,224,3))
x=aug(inp)
x=tf.keras.applications.mobilenet_v2.preprocess_input(x)
x=base(x,training=False)
x=layers.GlobalAveragePooling2D()(x)
x=layers.Dropout(0.2)(x)
out=layers.Dense(len(class_names),activation='softmax')(x)
model=models.Model(inp,out)
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
hist=model.fit(train_ds,validation_data=val_ds,epochs=10)
model.save('rice_leaf_model.keras')
